In [ ]:
#add libraries for tab
import pandas as pd #tab
import geopandas as gpd #tab
#mount the drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# import the geojson file '/content/drive/MyDrive/ESIIL2025MSU/Project Data/norm_irrigatedacres_gdf.geojson'
irrigatedacres_gdf = gpd.read_file('/content/drive/MyDrive/ESIIL2025MSU/Project Data/norm_irrigatedacres_gdf.geojson')

In [ ]:
# create a new dataframe with a column for 'YEAR' and columns for all unique counties in irrigatedacres_gdf

# Get the list of unique counties
all_counties = irrigatedacres_gdf['COUNTY'].unique()

# Populate the dataframe by iterating through the unique years in irrigatedacres_gdf
new_rows_all = []
for year in irrigatedacres_gdf['Year'].unique():
    year_data = irrigatedacres_gdf[irrigatedacres_gdf['Year'] == year]
    new_row_all = {'YEAR': year}
    for county in all_counties:
        county_data = year_data[year_data['COUNTY'] == county]
        if not county_data.empty:
            new_row_all[county] = county_data['norm_irrigatedacres'].iloc[0]
        else:
            new_row_all[county] = None # or pd.NA, depending on how you want to handle missing data
    new_rows_all.append(new_row_all)

nia_allcounties = pd.DataFrame(new_rows_all)
display(nia_allcounties.head())

#save the dataframe to a csv file
nia_allcounties.to_csv('/content/drive/MyDrive/ESIIL2025MSU/Project Data/nia_allcounties.csv', index=False)

,YEAR,LARIMER,LAS ANIMAS,FREMONT,GUNNISON,CONEJOS,EAGLE,OTERO,LA PLATA,SUMMIT,...,BROOMFIELD,WASHINGTON,ROUTT,ARCHULETA,GILPIN,DENVER,PARK,EL PASO,ARAPAHOE,HUERFANO
0,1982,6.197222,0.822011,1.049997,2.533449,14.552856,2.747841,7.578048,5.367808,1.548027,...,NaN,2.884518,3.121356,1.247999,NaN,0.089931,1.331884,0.877837,0.753309,1.101373
1,1987,4.735035,1.060075,1.235387,2.177130,12.641230,2.379504,6.921970,5.645002,2.640604,...,NaN,2.447941,2.930236,1.629527,NaN,0.009094,1.014075,0.966112,0.396927,1.380735
2,1992,4.904232,0.777256,1.301699,2.361929,14.191667,2.225261,7.434481,7.843173,1.185938,...,NaN,2.738559,3.052374,1.697587,NaN,NaN,0.806892,0.796973,0.642922,1.322020
3,1997,4.606092,0.785265,1.963091,2.463844,15.789868,1.536615,7.750525,6.599658,2.760207,...,NaN,3.439633,3.292163,1.930558,NaN,0.014146,1.272224,1.101424,0.756801,1.588737
4,2002,3.488109,0.174347,1.210328,1.975936,7.159559,0.859514,4.826163,6.121687,0.927050,...,NaN,2.686439,2.363868,0.865434,0.0,0.009094,0.765328,0.735628,0.627207,0.625477


In [ ]:
# prompt: I would like to use a .csv file in my google drive to complete an ANOVA analysis. I have a .csv file that includes data for 3 different counties with for the years 1982, 1987, 1992, 1997, 2002, 2007, 2012, 2017, and 2022. I would like an ANOVA analysis for the years 1985-2001, another for 2002-2022, and a third for all years 1985-2022. The first column is "YEAR", the second is "BACA", the third is "DOUGLAS", and the fourth is "EAGLE". create a code that will perform this analysis. The file I am using from my google drive is "nia_BacaDouglasEagle"

# Install necessary libraries
!pip install statsmodels

import pandas as pd
from google.colab import drive
import statsmodels.api as sm
from statsmodels.formula.api import ols
import numpy as np

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Load the CSV file
file_path = '/content/drive/MyDrive/ESIIL2025MSU/Project Data/nia_allcounties.csv'
df = pd.read_csv(file_path)

# Reshape the data for ANOVA (long format)
df_melted = df.melt(id_vars=['YEAR'], var_name='COUNTY', value_name='Normalized_Acres')

# ANOVA for years 1985-2001
df_1985_2001 = df_melted[(df_melted['YEAR'] >= 1985) & (df_melted['YEAR'] <= 2001)].copy()
# Drop rows with missing values for this period
df_1985_2001.dropna(inplace=True)

if not df_1985_2001.empty:
    print("ANOVA for years 1985-2001:")
    # Perform one-way ANOVA
    model_1985_2001 = ols('Normalized_Acres ~ C(COUNTY)', data=df_1985_2001).fit()
    anova_table_1985_2001 = sm.stats.anova_lm(model_1985_2001, typ=2)
    print(anova_table_1985_2001)
else:
    print("No data available for ANOVA for years 1985-2001 after dropping NaNs.")


# ANOVA for years 2002-2022
df_2002_2022 = df_melted[(df_melted['YEAR'] >= 2002) & (df_melted['YEAR'] <= 2022)].copy()
# Drop rows with missing values for this period
df_2002_2022.dropna(inplace=True)

if not df_2002_2022.empty:
    print("\nANOVA for years 2002-2022:")
    # Perform one-way ANOVA
    model_2002_2022 = ols('Normalized_Acres ~ C(COUNTY)', data=df_2002_2022).fit()
    anova_table_2002_2022 = sm.stats.anova_lm(model_2002_2022, typ=2)
    print(anova_table_2002_2022)
else:
    print("No data available for ANOVA for years 2002-2022 after dropping NaNs.")

# ANOVA for all years 1985-2022
df_1985_2022 = df_melted[(df_melted['YEAR'] >= 1985) & (df_melted['YEAR'] <= 2022)].copy()
# Drop rows with missing values for this period
df_1985_2022.dropna(inplace=True)

if not df_1985_2022.empty:
    print("\nANOVA for years 1985-2022:")
    # Perform one-way ANOVA
    model_1985_2022 = ols('Normalized_Acres ~ C(COUNTY)', data=df_1985_2022).fit()
    anova_table_1985_2022 = sm.stats.anova_lm(model_1985_2022, typ=2)
    print(anova_table_1985_2022)
else:
    print("No data available for ANOVA for years 1985-2022 after dropping NaNs.")


Mounted at /content/drive
ANOVA for years 1985-2001:
               sum_sq     df           F        PR(>F)
C(COUNTY)  5582.62931   60.0  147.087856  3.005377e-89
Residual     75.27620  119.0         NaN           NaN

ANOVA for years 2002-2022:
                sum_sq     df          F         PR(>F)
C(COUNTY)  6298.612476   63.0  92.416368  3.858650e-138
Residual    260.718882  241.0        NaN            NaN

ANOVA for years 1985-2022:
                 sum_sq     df           F         PR(>F)
C(COUNTY)  11702.216953   63.0  123.283303  4.272362e-234
Residual     634.315654  421.0         NaN            NaN


In [ ]:
# prompt: Use the To_csv() method to save each ANOVA table DataFrame to a CSV file with an appropriate filename

# Define the directory to save the CSV files
output_dir = '/content/drive/MyDrive/ESIIL2025MSU/Project Data/'

# Save the ANOVA tables to CSV files
if 'anova_table_1985_2001' in locals():
    anova_table_1985_2001.to_csv(output_dir + 'niaallcounty_anova_85_01.csv')

if 'anova_table_2002_2022' in locals():
    anova_table_2002_2022.to_csv(output_dir + 'niaallcounty_anova_02_22.csv')

if 'anova_table_1985_2022' in locals():
    anova_table_1985_2022.to_csv(output_dir + 'niaallcounty_anova_85_22.csv')

print("ANOVA tables saved to CSV files in", output_dir)


ANOVA tables saved to CSV files in /content/drive/MyDrive/ESIIL2025MSU/Project Data/
